# Customer Churn and Revenue Retention Analysis

## Notebook 2: Churn Modelling and Risk Scoring

This notebook builds a churn prediction model and converts model outputs into customer-level risk bands.

The aim is not only to predict whether a customer may churn, but to create a practical retention output that can be used in Power BI for business decision-making.

In [23]:
# Import required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

In [24]:
# Load the cleaned dataset

file_path = "../outputs/telco_customer_churn_cleaned.csv"

df = pd.read_csv(file_path)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnFlag,SeniorCitizenLabel,EstimatedAnnualRevenue,AverageMonthlyValue,TenureGroup,MonthlyChargeBand
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.8500,29.8500,No,0,Non-Senior,358.2000,29.8500,0-12 Months,Low Value
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.9500,"1,889.5000",No,0,Non-Senior,683.4000,55.5735,25-48 Months,Mid-Low Value
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.8500,108.1500,Yes,1,Non-Senior,646.2000,54.0750,0-12 Months,Mid-Low Value
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3000,"1,840.7500",No,0,Non-Senior,507.6000,40.9056,25-48 Months,Mid-Low Value
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7000,151.6500,Yes,1,Non-Senior,848.4000,75.8250,0-12 Months,Mid-High Value


## 1. Modelling Dataset Preparation

This section defines the target variable and selects features for churn modelling. Customer ID and the original text churn label are excluded because they do not provide predictive information for model training.

In [25]:
# Define target variable

target = "ChurnFlag"

# Remove columns that should not be used as model predictors
drop_columns = [
    "customerID",
    "Churn",
    "ChurnFlag"
]

X = df.drop(columns=drop_columns)
y = df[target]

X.shape, y.shape

((7043, 24), (7043,))

In [26]:
# Identify categorical and numeric columns

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features, numeric_features

C:\Users\jayap\AppData\Local\Temp\ipykernel_56768\428249898.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


(['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod',
  'SeniorCitizenLabel',
  'TenureGroup',
  'MonthlyChargeBand'],
 ['SeniorCitizen',
  'tenure',
  'MonthlyCharges',
  'TotalCharges',
  'EstimatedAnnualRevenue',
  'AverageMonthlyValue'])

## 2. Train-Test Split

The dataset is split into training and testing sets so that model performance can be evaluated on unseen customer records. Stratification is used to preserve the churn and non-churn balance in both sets.

In [27]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

((5282, 24),
 (1761, 24),
 np.float64(0.2654297614539947),
 np.float64(0.26519023282226006))

## 3. Preprocessing Pipeline

Categorical features are converted into numeric format using one-hot encoding. Numeric features are passed through unchanged. This allows the same preprocessing steps to be applied consistently during training and testing.

In [28]:
# Create preprocessing transformer

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

## 4. Baseline Model: Logistic Regression

Logistic Regression is used as a baseline model because it is interpretable and commonly used for binary classification problems such as churn prediction.

In [29]:
# Build Logistic Regression pipeline

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]
)

# Train model
logistic_model.fit(X_train, y_train)

# Predict on test set
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

In [30]:
# Build Logistic Regression pipeline

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]
)

# Train model
logistic_model.fit(X_train, y_train)

# Predict on test set
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

## 5. Logistic Regression Model Evaluation

The baseline Logistic Regression model is evaluated using accuracy, precision, recall, F1 score and ROC-AUC. For churn modelling, recall is especially important because the business wants to identify as many potential churners as possible.

In [31]:
# Create evaluation function

def evaluate_model(model_name, y_true, y_pred, y_prob):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

In [32]:
# Evaluate Logistic Regression

logistic_results = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_predictions,
    logistic_probabilities
)

logistic_results_df = pd.DataFrame([logistic_results])
logistic_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.7428,0.5096,0.7966,0.6216,0.8457


In [33]:
# Confusion matrix for Logistic Regression

logistic_confusion_matrix = confusion_matrix(y_test, logistic_predictions)

logistic_confusion_matrix

array([[936, 358],
       [ 95, 372]])

### Logistic Regression Model Insight

The baseline Logistic Regression model achieved 74.3% accuracy and a ROC-AUC score of 84.6%, indicating good separation between churned and retained customers.

The model achieved a recall score of 79.7%, meaning it identified most churned customers in the test set. This is useful for a retention-focused use case because the business would rather flag more at-risk customers than miss customers who are likely to leave.

However, precision was 51.0%, which means that some customers predicted as churn risks may not actually churn. In a business context, this is acceptable if retention actions are low-cost, such as targeted emails, service check-ins or contract upgrade offers.

## 6. Comparison Model: Random Forest

A Random Forest model is trained as a comparison model. It can capture non-linear relationships and interactions between customer features, although it is usually less directly interpretable than Logistic Regression.

In [34]:
# Build Random Forest pipeline

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

# Train model
random_forest_model.fit(X_train, y_train)

# Predict on test set
rf_predictions = random_forest_model.predict(X_test)
rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

In [35]:
# Evaluate Random Forest

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_predictions,
    rf_probabilities
)

rf_results_df = pd.DataFrame([rf_results])
rf_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Random Forest,0.7649,0.5382,0.7987,0.6431,0.8459


## 7. Hyperparameter Tuning

The Random Forest model is lightly tuned using cross-validation. The tuning objective is recall because the business priority is to identify as many potential churners as possible for retention action.

In [38]:
# Define parameter grid for Random Forest tuning

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [6, 8, 10],
    "model__min_samples_leaf": [5, 10, 20],
    "model__min_samples_split": [10, 20]
}

grid_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

grid_search.best_params_, grid_search.best_score_

Fitting 5 folds for each of 36 candidates, totalling 180 fits


({'model__max_depth': 6,
  'model__min_samples_leaf': 10,
  'model__min_samples_split': 10,
  'model__n_estimators': 200},
 np.float64(0.7952999491611592))

In [39]:
# Evaluate tuned Random Forest model

tuned_rf_model = grid_search.best_estimator_

tuned_rf_predictions = tuned_rf_model.predict(X_test)
tuned_rf_probabilities = tuned_rf_model.predict_proba(X_test)[:, 1]

tuned_rf_results = evaluate_model(
    "Tuned Random Forest",
    y_test,
    tuned_rf_predictions,
    tuned_rf_probabilities
)

tuned_rf_results_df = pd.DataFrame([tuned_rf_results])
tuned_rf_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Tuned Random Forest,0.7553,0.5250,0.8094,0.6369,0.8456


In [40]:
# Add tuned model to comparison table

final_model_comparison = pd.DataFrame([
    logistic_results,
    rf_results,
    tuned_rf_results
])

final_model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.7428,0.5096,0.7966,0.6216,0.8457
1,Random Forest,0.7649,0.5382,0.7987,0.6431,0.8459
2,Tuned Random Forest,0.7553,0.5250,0.8094,0.6369,0.8456


In [41]:
# Format final model comparison table

final_model_comparison_formatted = final_model_comparison.copy()

for col in metric_columns:
    final_model_comparison_formatted[col] = final_model_comparison_formatted[col].apply(lambda x: f"{x:.1%}")

final_model_comparison_formatted

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,74.3%,51.0%,79.7%,62.2%,84.6%
1,Random Forest,76.5%,53.8%,79.9%,64.3%,84.6%
2,Tuned Random Forest,75.5%,52.5%,80.9%,63.7%,84.6%


### Hyperparameter Tuning Insight

Hyperparameter tuning slightly improved the Random Forest model's recall from 79.9% to 80.9%, while reducing accuracy, precision and F1 score slightly. The ROC-AUC remained almost unchanged at 84.6%.

For this project, the tuned Random Forest is selected for customer-level risk scoring because the business objective is to identify as many potential churners as possible before they leave. This creates some additional false positives, but that trade-off is acceptable when retention actions are relatively low-cost, such as targeted emails, billing reviews, service check-ins or contract upgrade offers.

### Model Comparison Insight

Three models were compared: Logistic Regression, Random Forest and Tuned Random Forest.

Logistic Regression provided a strong interpretable baseline, achieving 74.3% accuracy, 79.7% recall and 84.6% ROC-AUC. The initial Random Forest model improved the balance of performance, achieving 76.5% accuracy, 53.8% precision, 79.9% recall and 64.3% F1 score.

After hyperparameter tuning, the Tuned Random Forest achieved 75.5% accuracy, 52.5% precision, 80.9% recall, 63.7% F1 score and 84.6% ROC-AUC. Although the tuned model slightly reduced accuracy, precision and F1 score, it improved recall.

For this project, the Tuned Random Forest is selected as the final model for customer-level churn risk scoring because the business objective is to identify as many potential churners as possible before they leave. In a retention context, a higher recall is valuable because missing likely churners can result in lost recurring revenue. The lower precision is acceptable if the business uses low-cost retention actions such as targeted emails, billing reviews, service check-ins or contract upgrade offers.

## 8. Customer-Level Churn Risk Scoring

The selected tuned Random Forest model is used to generate churn probabilities for every customer in the dataset. These probabilities are converted into business-friendly risk bands so that the output can be used for retention prioritisation and Power BI reporting.

In [42]:
# Generate churn probabilities for all customers using the tuned Random Forest model

customer_risk_scores = df.copy()

customer_risk_scores["PredictedChurnProbability"] = tuned_rf_model.predict_proba(X)[:, 1]

customer_risk_scores[
    [
        "customerID",
        "Contract",
        "TenureGroup",
        "InternetService",
        "PaymentMethod",
        "MonthlyCharges",
        "EstimatedAnnualRevenue",
        "Churn",
        "PredictedChurnProbability"
    ]
].head()

,customerID,Contract,TenureGroup,InternetService,PaymentMethod,MonthlyCharges,EstimatedAnnualRevenue,Churn,PredictedChurnProbability
0,7590-VHVEG,Month-to-month,0-12 Months,DSL,Electronic check,29.8500,358.2000,No,0.7168
1,5575-GNVDE,One year,25-48 Months,DSL,Mailed check,56.9500,683.4000,No,0.1809
2,3668-QPYBK,Month-to-month,0-12 Months,DSL,Mailed check,53.8500,646.2000,Yes,0.6099
3,7795-CFOCW,One year,25-48 Months,DSL,Bank transfer (automatic),42.3000,507.6000,No,0.1434
4,9237-HQITU,Month-to-month,0-12 Months,Fiber optic,Electronic check,70.7000,848.4000,Yes,0.8509


In [43]:
# Create customer churn risk bands

def assign_risk_band(probability):
    if probability >= 0.75:
        return "Critical Risk"
    elif probability >= 0.50:
        return "High Risk"
    elif probability >= 0.25:
        return "Medium Risk"
    else:
        return "Low Risk"

customer_risk_scores["ChurnRiskBand"] = customer_risk_scores["PredictedChurnProbability"].apply(assign_risk_band)

customer_risk_scores["ChurnRiskBand"].value_counts()

ChurnRiskBand
Low Risk         2456
High Risk        1892
Medium Risk      1679
Critical Risk    1016
Name: count, dtype: int64

## 9. Revenue Exposure by Churn Risk Band

Risk bands are analysed by customer count, actual churn rate and estimated annual revenue exposure. This helps translate model output into business retention priorities.

In [45]:
# Analyse churn risk bands by customer count, churn rate and revenue exposure

risk_band_summary = (
    customer_risk_scores.groupby("ChurnRiskBand")
    .agg(
        Customers=("customerID", "nunique"),
        ActualChurnedCustomers=("ChurnFlag", "sum"),
        ActualChurnRate=("ChurnFlag", "mean"),
        TotalEstimatedAnnualRevenue=("EstimatedAnnualRevenue", "sum"),
        AveragePredictedChurnProbability=("PredictedChurnProbability", "mean")
    )
    .reset_index()
)

risk_band_order = ["Low Risk", "Medium Risk", "High Risk", "Critical Risk"]

risk_band_summary["ChurnRiskBand"] = pd.Categorical(
    risk_band_summary["ChurnRiskBand"],
    categories=risk_band_order,
    ordered=True
)

risk_band_summary = risk_band_summary.sort_values("ChurnRiskBand")

risk_band_summary

,ChurnRiskBand,Customers,ActualChurnedCustomers,ActualChurnRate,TotalEstimatedAnnualRevenue,AveragePredictedChurnProbability
2,Low Risk,2456,63,0.0257,"1,477,028.4000",0.1127
3,Medium Risk,1679,260,0.1549,"1,389,825.0000",0.3709
1,High Risk,1892,816,0.4313,"1,624,153.2000",0.6302
0,Critical Risk,1016,730,0.7185,"982,392.6000",0.8323


In [46]:
# Format risk band summary for reporting

risk_band_summary_formatted = risk_band_summary.copy()

risk_band_summary_formatted["ActualChurnRate"] = risk_band_summary_formatted["ActualChurnRate"].apply(lambda x: f"{x:.1%}")
risk_band_summary_formatted["TotalEstimatedAnnualRevenue"] = risk_band_summary_formatted["TotalEstimatedAnnualRevenue"].apply(lambda x: f"£{x:,.2f}")
risk_band_summary_formatted["AveragePredictedChurnProbability"] = risk_band_summary_formatted["AveragePredictedChurnProbability"].apply(lambda x: f"{x:.1%}")

risk_band_summary_formatted

,ChurnRiskBand,Customers,ActualChurnedCustomers,ActualChurnRate,TotalEstimatedAnnualRevenue,AveragePredictedChurnProbability
2,Low Risk,2456,63,2.6%,"£1,477,028.40",11.3%
3,Medium Risk,1679,260,15.5%,"£1,389,825.00",37.1%
1,High Risk,1892,816,43.1%,"£1,624,153.20",63.0%
0,Critical Risk,1016,730,71.9%,"£982,392.60",83.2%


### Risk Band Insight

The churn risk bands show a clear separation between customer groups. Low-risk customers have an actual churn rate of only 2.6%, while critical-risk customers have an actual churn rate of 71.9%. This confirms that the model-generated risk bands are meaningful for business prioritisation.

The High Risk segment contains 1,892 customers and carries the largest estimated annual revenue exposure at approximately £1.62m. The Critical Risk segment has the highest churn concentration, with 730 churned customers out of 1,016 and an actual churn rate of 71.9%.

From a retention perspective, the business should treat Critical Risk customers as requiring urgent intervention, while High Risk customers should be targeted through proactive retention campaigns before they move into the critical stage.

In [47]:
# Assign recommended retention actions by risk band

def assign_retention_action(risk_band):
    if risk_band == "Critical Risk":
        return "Urgent retention call, service review, personalised offer"
    elif risk_band == "High Risk":
        return "Proactive retention campaign and contract upgrade incentive"
    elif risk_band == "Medium Risk":
        return "Targeted engagement, onboarding support and usage monitoring"
    else:
        return "Maintain standard service and monitor customer experience"

customer_risk_scores["RecommendedRetentionAction"] = customer_risk_scores["ChurnRiskBand"].apply(assign_retention_action)

customer_risk_scores[
    [
        "customerID",
        "ChurnRiskBand",
        "PredictedChurnProbability",
        "EstimatedAnnualRevenue",
        "RecommendedRetentionAction"
    ]
].head(10)

,customerID,ChurnRiskBand,PredictedChurnProbability,EstimatedAnnualRevenue,RecommendedRetentionAction
0,7590-VHVEG,High Risk,0.7168,358.2000,Proactive retention campaign and contract upgr...
1,5575-GNVDE,Low Risk,0.1809,683.4000,Maintain standard service and monitor customer...
2,3668-QPYBK,High Risk,0.6099,646.2000,Proactive retention campaign and contract upgr...
3,7795-CFOCW,Low Risk,0.1434,507.6000,Maintain standard service and monitor customer...
4,9237-HQITU,Critical Risk,0.8509,848.4000,"Urgent retention call, service review, persona..."
5,9305-CDSKC,Critical Risk,0.8848,"1,195.8000","Urgent retention call, service review, persona..."
6,1452-KIOVK,High Risk,0.6733,"1,069.2000",Proactive retention campaign and contract upgr...
7,6713-OKOMC,High Risk,0.5160,357.0000,Proactive retention campaign and contract upgr...
8,7892-POOKP,High Risk,0.6893,"1,257.6000",Proactive retention campaign and contract upgr...
9,6388-TABGU,Low Risk,0.1545,673.8000,Maintain standard service and monitor customer...


## 10. Save Customer Risk Scoring Output

The final customer-level risk scoring dataset is saved for Power BI reporting and retention analysis.

In [48]:
# Select final columns for Power BI and business reporting

risk_scoring_output = customer_risk_scores[
    [
        "customerID",
        "gender",
        "SeniorCitizenLabel",
        "Partner",
        "Dependents",
        "tenure",
        "TenureGroup",
        "Contract",
        "InternetService",
        "PaymentMethod",
        "MonthlyCharges",
        "TotalCharges",
        "EstimatedAnnualRevenue",
        "AverageMonthlyValue",
        "MonthlyChargeBand",
        "Churn",
        "ChurnFlag",
        "PredictedChurnProbability",
        "ChurnRiskBand",
        "RecommendedRetentionAction"
    ]
].copy()

risk_scoring_output.head()

,customerID,gender,SeniorCitizenLabel,Partner,Dependents,tenure,TenureGroup,Contract,InternetService,PaymentMethod,MonthlyCharges,TotalCharges,EstimatedAnnualRevenue,AverageMonthlyValue,MonthlyChargeBand,Churn,ChurnFlag,PredictedChurnProbability,ChurnRiskBand,RecommendedRetentionAction
0,7590-VHVEG,Female,Non-Senior,Yes,No,1,0-12 Months,Month-to-month,DSL,Electronic check,29.8500,29.8500,358.2000,29.8500,Low Value,No,0,0.7168,High Risk,Proactive retention campaign and contract upgr...
1,5575-GNVDE,Male,Non-Senior,No,No,34,25-48 Months,One year,DSL,Mailed check,56.9500,"1,889.5000",683.4000,55.5735,Mid-Low Value,No,0,0.1809,Low Risk,Maintain standard service and monitor customer...
2,3668-QPYBK,Male,Non-Senior,No,No,2,0-12 Months,Month-to-month,DSL,Mailed check,53.8500,108.1500,646.2000,54.0750,Mid-Low Value,Yes,1,0.6099,High Risk,Proactive retention campaign and contract upgr...
3,7795-CFOCW,Male,Non-Senior,No,No,45,25-48 Months,One year,DSL,Bank transfer (automatic),42.3000,"1,840.7500",507.6000,40.9056,Mid-Low Value,No,0,0.1434,Low Risk,Maintain standard service and monitor customer...
4,9237-HQITU,Female,Non-Senior,No,No,2,0-12 Months,Month-to-month,Fiber optic,Electronic check,70.7000,151.6500,848.4000,75.8250,Mid-High Value,Yes,1,0.8509,Critical Risk,"Urgent retention call, service review, persona..."


In [49]:
# Save final risk scoring output

risk_scoring_file_path = "../outputs/customer_churn_risk_scores.csv"

risk_scoring_output.to_csv(risk_scoring_file_path, index=False)

risk_scoring_file_path

'../outputs/customer_churn_risk_scores.csv'

In [50]:
# Confirm output shape

risk_scoring_output.shape

(7043, 20)